# Getting started

A minimal tour of `sam2-stream`: build a predictor, then track an object through a stream one frame
at a time. This notebook uses a tiny synthetic clip so it runs without any data; swap in your webcam
or a video file with the scripts in `examples/`.

Prerequisites: SAM 2 installed with a checkpoint, and `pip install -e .` for this package (see the
README).

## Configure the model

In [ ]:
# point these at your SAM 2 config + checkpoint (config path is resolved by SAM 2's hydra setup)
CFG  = "configs/sam2.1/sam2.1_hiera_t.yaml"
CKPT = "checkpoints/sam2.1_hiera_tiny.pt"
DEVICE = "cuda"

In [ ]:
import numpy as np, cv2, torch
from sam2.build_sam import build_sam2_video_predictor
from sam2_stream import LiveSAM2, overlay_masks

predictor = build_sam2_video_predictor(CFG, CKPT, device=DEVICE)
live = LiveSAM2(predictor)          # ring buffer + memory eviction handled internally

## A self-contained demo

We make a short clip of a moving square on a fixed background, seed a box around it on the first
frame, and let `sam2-stream` track it. `live.track(...)` yields one result per frame and pulls the
next frame from the callable we give it.

In [ ]:
H, W, N = 360, 640, 60
_bg = np.tile(np.linspace(30, 95, W, dtype=np.uint8), (H, 1))[..., None].repeat(3, axis=2)

def make_frame(i):
    img = _bg.copy()
    x = 40 + i * 8                                  # square drifts to the right
    cv2.rectangle(img, (x, 150), (x + 80, 230), (30, 170, 220), -1)
    return img

frames = [make_frame(i) for i in range(N)]
prompts = [{"obj_id": 1, "kind": "box", "box": (40, 150, 120, 230)}]   # box around the square on frame 0

it = iter(range(1, N))
def next_frame():
    i = next(it, None)
    return frames[i] if i is not None else None

results = []
for frame_idx, frame_rgb, masks in live.track(frames[0], prompts, next_frame):
    results.append(overlay_masks(frame_rgb, masks))
print(f"tracked {len(results)} frames")

## Look at a few frames

In [ ]:
import matplotlib.pyplot as plt
picks = [0, len(results)//3, 2*len(results)//3, len(results)-1]
fig, axes = plt.subplots(1, len(picks), figsize=(16, 4))
for ax, i in zip(axes, picks):
    ax.imshow(cv2.cvtColor(results[i], cv2.COLOR_BGR2RGB)); ax.set_title(f"frame {i}"); ax.axis("off")
plt.tight_layout()

## Next

- **Live webcam:** `python examples/track_camera.py --cfg ... --ckpt ... --cam 0`
- **A video file (any length):** `python examples/track_video.py --cfg ... --ckpt ... --video in.mp4 --box X1 Y1 X2 Y2 --out out.mp4`

For multiple objects, add more entries to `prompts`, each with its own `obj_id` and a `point` or
`box`. See `docs/how-it-works.md` for why memory stays flat.